# 🧵 DIAN Fabric 이미지 업스케일 (Real-ESRGAN)

600px 미만 원단 이미지를 4x 업스케일하여 Supabase에 재업로드

**사용 전 준비:**
1. 런타임 > 런타임 유형 변경 > GPU 선택
2. 아래 셀을 순서대로 실행
3. SUPABASE_URL, SUPABASE_KEY, GEMINI_API_KEY 입력

In [ ]:
# 1. Real-ESRGAN 설치
!pip install -q realesrgan basicsr gfpgan
!pip install -q supabase
print('✅ 설치 완료')

In [ ]:
# 2. 환경 변수 설정 (여기에 직접 입력하세요)
SUPABASE_URL = 'https://qkkobestkhkxlrjeuakt.supabase.co'
SUPABASE_KEY = ''  # ← .env.local의 SUPABASE_SERVICE_KEY 입력

if not SUPABASE_KEY:
    raise ValueError('❌ SUPABASE_KEY를 입력해주세요!')
print('✅ 환경 변수 설정 완료')

In [ ]:
# 3. Supabase에서 600px 미만 이미지 목록 가져오기
import requests
from io import BytesIO
from PIL import Image
import json

headers = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type': 'application/json'
}

# 전체 원단 목록 가져오기
all_fabrics = []
offset = 0
while True:
    url = f"{SUPABASE_URL}/rest/v1/fabrics?select=id,name,color_code,image_url&image_url=not.is.null&offset={offset}&limit=1000"
    res = requests.get(url, headers=headers)
    data = res.json()
    if not data:
        break
    all_fabrics.extend(data)
    offset += 1000

print(f'총 원단: {len(all_fabrics)}개')

# 해상도 체크 + 600px 미만 필터링
import concurrent.futures

needs_upscale = []
checked = 0

def check_resolution(fabric):
    try:
        res = requests.get(fabric['image_url'], timeout=10)
        img = Image.open(BytesIO(res.content))
        w, h = img.size
        min_dim = min(w, h)
        return (fabric, min_dim, w, h) if min_dim < 600 else None
    except:
        return None

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(check_resolution, f): f for f in all_fabrics}
    for future in concurrent.futures.as_completed(futures):
        checked += 1
        result = future.result()
        if result:
            needs_upscale.append(result)
        if checked % 500 == 0:
            print(f'{checked}/{len(all_fabrics)} 체크 완료... ({len(needs_upscale)}개 업스케일 필요)')

print(f'\n✅ 체크 완료: {len(needs_upscale)}개 업스케일 대상')

In [ ]:
# 4. Real-ESRGAN 모델 로드
import torch
import numpy as np
import cv2
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)

upsampler = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    model=model,
    tile=400,
    tile_pad=10,
    pre_pad=0,
    half=True
)

print(f'✅ Real-ESRGAN 모델 로드 완료 (GPU: {torch.cuda.is_available()})')

In [ ]:
# 5. 업스케일 + Supabase 재업로드 + 목록 저장
import os
import time
import csv

success = 0
errors = 0
start_time = time.time()

# 업스케일 로그 기록
upscale_log = []

for i, (fabric, min_dim, orig_w, orig_h) in enumerate(needs_upscale):
    try:
        # 이미지 다운로드
        res = requests.get(fabric['image_url'], timeout=10)
        img_array = np.frombuffer(res.content, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        if img is None:
            errors += 1
            upscale_log.append({
                'name': fabric['name'],
                'color_code': fabric['color_code'],
                'status': 'ERROR',
                'reason': 'decode failed',
                'before': f'{orig_w}x{orig_h}',
                'after': '-'
            })
            continue

        # 업스케일
        output, _ = upsampler.enhance(img, outscale=4)

        # 너무 크면 1200px로 리사이즈
        h, w = output.shape[:2]
        if max(w, h) > 1200:
            scale = 1200 / max(w, h)
            output = cv2.resize(output, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LANCZOS4)

        new_h, new_w = output.shape[:2]

        # JPEG 인코딩
        _, buffer = cv2.imencode('.jpeg', output, [cv2.IMWRITE_JPEG_QUALITY, 85])
        img_bytes = buffer.tobytes()

        # Supabase Storage 경로 추출
        storage_path = fabric['image_url'].split('/Fabric-images/')[-1]

        # Supabase에 업로드 (덮어쓰기)
        upload_url = f"{SUPABASE_URL}/storage/v1/object/Fabric-images/{storage_path}"
        upload_headers = {
            'Authorization': f'Bearer {SUPABASE_KEY}',
            'Content-Type': 'image/jpeg',
            'x-upsert': 'true'
        }
        upload_res = requests.put(upload_url, headers=upload_headers, data=img_bytes)

        if upload_res.status_code in [200, 201]:
            success += 1
            upscale_log.append({
                'name': fabric['name'],
                'color_code': fabric['color_code'],
                'status': 'OK',
                'reason': '',
                'before': f'{orig_w}x{orig_h}',
                'after': f'{new_w}x{new_h}',
                'size_kb': round(len(img_bytes) / 1024)
            })
        else:
            errors += 1
            upscale_log.append({
                'name': fabric['name'],
                'color_code': fabric['color_code'],
                'status': 'UPLOAD_ERROR',
                'reason': f'{upload_res.status_code}',
                'before': f'{orig_w}x{orig_h}',
                'after': f'{new_w}x{new_h}'
            })

    except Exception as e:
        errors += 1
        upscale_log.append({
            'name': fabric['name'],
            'color_code': fabric.get('color_code', ''),
            'status': 'ERROR',
            'reason': str(e)[:80],
            'before': f'{orig_w}x{orig_h}',
            'after': '-'
        })

    if (i + 1) % 50 == 0 or i + 1 == len(needs_upscale):
        elapsed = (time.time() - start_time) / 60
        remaining = elapsed / (i + 1) * (len(needs_upscale) - i - 1)
        print(f'  [{i+1}/{len(needs_upscale)}] {elapsed:.1f}분 경과, ~{remaining:.0f}분 남음 | 성공:{success} 에러:{errors}')

# CSV 로그 저장
csv_path = 'upscale_log.csv'
with open(csv_path, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'color_code', 'status', 'before', 'after', 'size_kb', 'reason'])
    writer.writeheader()
    writer.writerows(upscale_log)

print(f'\n🎉 업스케일 완료!')
print(f'  성공: {success}개')
print(f'  에러: {errors}개')
print(f'  총 시간: {(time.time() - start_time) / 60:.1f}분')
print(f'  로그 저장: {csv_path}')
print(f'\n⬇️ 아래에서 upscale_log.csv 다운로드하세요')

In [ ]:
# 6. 로그 파일 다운로드
from google.colab import files
files.download('upscale_log.csv')
print('✅ upscale_log.csv 다운로드 완료 — 엑셀에서 확인하세요')